# 33 — 統計学と機械学習の橋渡し

このノートブックは **statspg**（統計検定準1級向け）と **mlpg**（ML実践向け）の橋渡しです。  
統計学の基礎概念が、機械学習アルゴリズムのどこに対応しているかを体系的に解説します。

## 目次
1. 線形回帰（統計） → 線形モデル（ML）
2. 最尤推定（MLE） → 損失関数最小化
3. ベイズ推定 → 正則化（Ridge / LASSO）
4. 仮説検定 → モデル評価指標
5. 総まとめ：統計 ↔ ML 対応表

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('ライブラリのインポート完了')

---
## 1. 線形回帰（統計） → 線形モデル（ML）

### 1.1 統計学の視点：回帰モデル

統計学では回帰モデルを次のように定式化します：

$$y_i = \beta_0 + \beta_1 x_{i1} + \cdots + \beta_p x_{ip} + \varepsilon_i, \quad \varepsilon_i \sim \mathcal{N}(0, \sigma^2)$$

**最小二乗法（OLS）** で $\hat{\boldsymbol{\beta}} = (X^\top X)^{-1} X^\top y$ を求めます。

### 1.2 MLの視点：予測モデル

MLでは同じ問題を「**損失関数を最小化するパラメータを求める最適化問題**」として捉えます：

$$\hat{\boldsymbol{\beta}} = \arg\min_{\boldsymbol{\beta}} \underbrace{\sum_{i=1}^n (y_i - \hat{y}_i)^2}_{\text{MSE損失（＝残差平方和）}}$$

**両者は数学的に同一です！** 違いは「解釈の枠組み」です。

| 観点 | 統計学 | 機械学習 |
|------|--------|----------|
| 目的 | 母集団のパラメータ推定・推測 | 未知データへの予測精度最大化 |
| 評価 | 有意性（p値）・信頼区間 | MSE・$R^2$・汎化誤差 |
| 解釈 | 係数の意味・因果関係 | ブラックボックスでも予測精度重視 |

In [ ]:
# データ生成
n = 100
X_raw = np.random.randn(n, 2)
true_beta = np.array([1.5, -0.8])
y = 2.0 + X_raw @ true_beta + np.random.randn(n) * 0.5

X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

# === 統計学の方法：scipy / statsmodels 風の手計算 OLS ===
X_aug = np.column_stack([np.ones(len(X_train)), X_train])  # 切片列を追加
beta_ols = np.linalg.lstsq(X_aug, y_train, rcond=None)[0]

y_pred_stat = np.column_stack([np.ones(len(X_test)), X_test]) @ beta_ols
mse_stat = mean_squared_error(y_test, y_pred_stat)

# === MLの方法：scikit-learn LinearRegression ===
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_ml = lr.predict(X_test)
mse_ml = mean_squared_error(y_test, y_pred_ml)

print('=== 推定係数の比較 ===')
print(f'真の値      : β0={2.0:.4f}, β1={true_beta[0]:.4f}, β2={true_beta[1]:.4f}')
print(f'統計(OLS)   : β0={beta_ols[0]:.4f}, β1={beta_ols[1]:.4f}, β2={beta_ols[2]:.4f}')
print(f'ML(sklearn) : β0={lr.intercept_:.4f}, β1={lr.coef_[0]:.4f}, β2={lr.coef_[1]:.4f}')
print(f'\nテストMSE 統計: {mse_stat:.4f}, ML: {mse_ml:.4f}  ← 同じ！')

In [ ]:
# statsmodels で統計的解釈を追加
try:
    import statsmodels.api as sm
    model_sm = sm.OLS(y_train, sm.add_constant(X_train)).fit()
    print(model_sm.summary().tables[1])
    print('\n統計学では「係数が有意か（p<0.05か）」を重視します')
    print('MLでは「テストセットの予測誤差」を重視します')
except ImportError:
    print('statsmodels がインストールされていません。係数の有意性確認は省略します。')
    print(f'\nOLS係数: β0={beta_ols[0]:.4f}, β1={beta_ols[1]:.4f}, β2={beta_ols[2]:.4f}')
    print('（statsmodels があれば p値・信頼区間を確認できます）')

In [ ]:
# 統計とMLの評価指標の対応を可視化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 残差プロット（統計で重視）
residuals = y_train - (np.column_stack([np.ones(len(X_train)), X_train]) @ beta_ols)
axes[0].scatter(np.column_stack([np.ones(len(X_train)), X_train]) @ beta_ols,
                residuals, alpha=0.6, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('予測値 $\\hat{y}$')
axes[0].set_ylabel('残差 $y - \\hat{y}$')
axes[0].set_title('残差プロット\n（統計：誤差の正規性・等分散性を確認）')

# 予測 vs 実測（MLで重視）
axes[1].scatter(y_test, y_pred_ml, alpha=0.7, color='darkorange')
lo, hi = min(y_test.min(), y_pred_ml.min()), max(y_test.max(), y_pred_ml.max())
axes[1].plot([lo, hi], [lo, hi], 'r--', label='完全予測')
axes[1].set_xlabel('実測値 $y$')
axes[1].set_ylabel('予測値 $\\hat{y}$')
r2 = r2_score(y_test, y_pred_ml)
axes[1].set_title(f'予測 vs 実測\n（ML：汎化誤差 MSE={mse_ml:.3f}, R²={r2:.3f}）')
axes[1].legend()

plt.suptitle('統計とMLでの回帰の見方', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bridge_regression.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 2. 最尤推定（MLE） → 損失関数最小化

### 2.1 最尤推定とは

観測データ $\mathcal{D} = \{(x_i, y_i)\}$ が得られたとき、それを「最も尤もらしく」説明するパラメータ $\boldsymbol{\theta}$ を求める方法です：

$$\hat{\boldsymbol{\theta}}_{\text{MLE}} = \arg\max_{\boldsymbol{\theta}} \prod_{i=1}^n p(y_i \mid x_i; \boldsymbol{\theta})$$

対数をとると（対数尤度最大化）：

$$\hat{\boldsymbol{\theta}}_{\text{MLE}} = \arg\max_{\boldsymbol{\theta}} \sum_{i=1}^n \log p(y_i \mid x_i; \boldsymbol{\theta})$$

### 2.2 MLE と損失関数の等価性

| モデル | 誤差分布 | MLE最大化 | ≡ ML損失最小化 |
|--------|---------|-----------|---------------|
| 線形回帰 | $y \sim \mathcal{N}(\hat{y}, \sigma^2)$ | 対数尤度最大 | MSE最小 |
| ロジスティック回帰 | $y \sim \text{Bernoulli}(\sigma(\hat{y}))$ | 対数尤度最大 | 交差エントロピー最小 |
| ポアソン回帰 | $y \sim \text{Poisson}(e^{\hat{y}})$ | 対数尤度最大 | ポアソン偏差最小 |

In [ ]:
# ロジスティック回帰：MLE = 交差エントロピー最小化を示す

# 2クラス分類データ
n_cls = 200
X_cls = np.random.randn(n_cls, 2)
# 線形境界で分離できるクラス
y_cls = (X_cls[:, 0] * 0.8 + X_cls[:, 1] * 0.6 + np.random.randn(n_cls) * 0.3 > 0).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y_cls, test_size=0.25, random_state=42)

# ロジスティック回帰（sklearn は内部で MLE = 交差エントロピー最小化）
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_tr, y_tr)

# MLE として手動で対数尤度を計算
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def log_likelihood(X, y, w, b):
    z = X @ w + b
    p = sigmoid(z)
    return np.mean(y * np.log(p + 1e-15) + (1 - y) * np.log(1 - p + 1e-15))

def cross_entropy_loss(X, y, w, b):
    return -log_likelihood(X, y, w, b)

ll = log_likelihood(X_te, y_te, log_reg.coef_[0], log_reg.intercept_[0])
ce = cross_entropy_loss(X_te, y_te, log_reg.coef_[0], log_reg.intercept_[0])

print('=== MLE と 交差エントロピー損失の関係 ===')
print(f'対数尤度 (MLE最大化):  {ll:.4f}')
print(f'交差エントロピー損失:  {ce:.4f}  ← -1 × 対数尤度')
print(f'\n→ 「対数尤度を最大化」 ≡ 「交差エントロピー損失を最小化」')
print(f'\nROC AUC: {roc_auc_score(y_te, log_reg.predict_proba(X_te)[:, 1]):.4f}')

In [ ]:
# 対数尤度 vs 交差エントロピー損失を可視化
# パラメータを1次元で変化させてプロット
w_opt = log_reg.coef_[0].copy()
b_opt = log_reg.intercept_[0]

scale_range = np.linspace(0.2, 2.0, 100)
log_likes = [log_likelihood(X_te, y_te, w_opt * s, b_opt) for s in scale_range]
ce_losses  = [-ll for ll in log_likes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(scale_range, log_likes, color='steelblue', linewidth=2)
axes[0].axvline(1.0, color='red', linestyle='--', label='最適 w')
axes[0].set_xlabel('重みのスケール係数')
axes[0].set_ylabel('対数尤度')
axes[0].set_title('最尤推定（MLE）\n対数尤度を最大化')
axes[0].legend()

axes[1].plot(scale_range, ce_losses, color='darkorange', linewidth=2)
axes[1].axvline(1.0, color='red', linestyle='--', label='最適 w')
axes[1].set_xlabel('重みのスケール係数')
axes[1].set_ylabel('交差エントロピー損失')
axes[1].set_title('機械学習（ML）\n交差エントロピー損失を最小化')
axes[1].legend()

plt.suptitle('MLE の最大化 ≡ 交差エントロピー損失の最小化', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bridge_mle_loss.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 3. ベイズ推定 → 正則化（Ridge / LASSO）

### 3.1 事後分布の最大化（MAP推定）

ベイズ推定では、パラメータに**事前分布**を置き、事後分布を最大化します（MAP: Maximum A Posteriori）：

$$\hat{\boldsymbol{\theta}}_{\text{MAP}} = \arg\max_{\boldsymbol{\theta}} \underbrace{p(\boldsymbol{\theta} \mid \mathcal{D})}_{\text{事後分布}} \propto \underbrace{p(\mathcal{D} \mid \boldsymbol{\theta})}_{\text{尤度}} \cdot \underbrace{p(\boldsymbol{\theta})}_{\text{事前分布}}$$

### 3.2 事前分布 と 正則化の等価性

$$\log p(\boldsymbol{\theta} \mid \mathcal{D}) = \log p(\mathcal{D} \mid \boldsymbol{\theta}) + \log p(\boldsymbol{\theta}) + \text{const}$$

| ベイズ事前分布 | MAP推定 | ≡ ML正則化 |
|---------------|--------|------------|
| $\beta_j \sim \mathcal{N}(0, 1/\lambda)$ | MAP最大化 | **Ridge** ($L_2$正則化): $+ \lambda \|\boldsymbol{\beta}\|_2^2$ |
| $\beta_j \sim \text{Laplace}(0, 1/\lambda)$ | MAP最大化 | **LASSO** ($L_1$正則化): $+ \lambda \|\boldsymbol{\beta}\|_1$ |

Ridge は係数を**縮小（shrinkage）**し、LASSO は係数を**スパース（0に圧縮）**にします。

In [ ]:
# 多重共線性あり・特徴量多めのデータで Ridge vs LASSO を比較
n_reg = 150
p_total = 15   # 特徴量数
p_true  = 5    # 真に有効な特徴量数

X_reg = np.random.randn(n_reg, p_total)
true_coef = np.zeros(p_total)
true_coef[:p_true] = np.random.uniform(0.5, 2.0, p_true) * np.random.choice([-1, 1], p_true)

y_reg = X_reg @ true_coef + np.random.randn(n_reg) * 0.5

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 正則化強度を変えてテストMSEを計算
alphas = np.logspace(-3, 2, 50)

ridge_mses, lasso_mses = [], []
for a in alphas:
    ridge = Ridge(alpha=a).fit(X_tr_r, y_tr_r)
    lasso = Lasso(alpha=a, max_iter=5000).fit(X_tr_r, y_tr_r)
    ridge_mses.append(mean_squared_error(y_te_r, ridge.predict(X_te_r)))
    lasso_mses.append(mean_squared_error(y_te_r, lasso.predict(X_te_r)))

best_ridge_alpha = alphas[np.argmin(ridge_mses)]
best_lasso_alpha = alphas[np.argmin(lasso_mses)]

print(f'最適 Ridge alpha = {best_ridge_alpha:.4f}  MSE = {min(ridge_mses):.4f}')
print(f'最適 LASSO alpha = {best_lasso_alpha:.4f}  MSE = {min(lasso_mses):.4f}')

In [ ]:
# 係数パスと正則化効果の可視化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Ridge 係数パス
ridge_coefs = [Ridge(alpha=a).fit(X_tr_r, y_tr_r).coef_ for a in alphas]
ridge_coefs = np.array(ridge_coefs)
for j in range(p_total):
    color = 'steelblue' if j < p_true else 'lightgray'
    lw = 2 if j < p_true else 0.8
    axes[0, 0].semilogx(alphas, ridge_coefs[:, j], color=color, linewidth=lw)
axes[0, 0].axvline(best_ridge_alpha, color='red', linestyle='--', alpha=0.7)
axes[0, 0].set_xlabel('正則化強度 λ')
axes[0, 0].set_ylabel('係数値')
axes[0, 0].set_title('Ridge 係数パス\n（L₂正則化：ガウス事前分布に対応）')

# LASSO 係数パス
lasso_coefs = [Lasso(alpha=a, max_iter=5000).fit(X_tr_r, y_tr_r).coef_ for a in alphas]
lasso_coefs = np.array(lasso_coefs)
for j in range(p_total):
    color = 'darkorange' if j < p_true else 'lightgray'
    lw = 2 if j < p_true else 0.8
    axes[0, 1].semilogx(alphas, lasso_coefs[:, j], color=color, linewidth=lw)
axes[0, 1].axvline(best_lasso_alpha, color='red', linestyle='--', alpha=0.7)
axes[0, 1].set_xlabel('正則化強度 λ')
axes[0, 1].set_ylabel('係数値')
axes[0, 1].set_title('LASSO 係数パス\n（L₁正則化：ラプラス事前分布に対応）')

# テストMSE 比較
axes[1, 0].semilogx(alphas, ridge_mses, label='Ridge', color='steelblue', linewidth=2)
axes[1, 0].semilogx(alphas, lasso_mses, label='LASSO', color='darkorange', linewidth=2)
axes[1, 0].axvline(best_ridge_alpha, color='steelblue', linestyle='--', alpha=0.5)
axes[1, 0].axvline(best_lasso_alpha, color='darkorange', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('正則化強度 λ')
axes[1, 0].set_ylabel('テスト MSE')
axes[1, 0].set_title('正則化強度 vs テスト MSE')
axes[1, 0].legend()

# 推定係数の比較（最適 α での）
ridge_best = Ridge(alpha=best_ridge_alpha).fit(X_tr_r, y_tr_r)
lasso_best = Lasso(alpha=best_lasso_alpha, max_iter=5000).fit(X_tr_r, y_tr_r)
x_pos = np.arange(p_total)
width = 0.25
axes[1, 1].bar(x_pos - width, true_coef, width, label='真の係数', color='green', alpha=0.7)
axes[1, 1].bar(x_pos, ridge_best.coef_, width, label='Ridge', color='steelblue', alpha=0.7)
axes[1, 1].bar(x_pos + width, lasso_best.coef_, width, label='LASSO', color='darkorange', alpha=0.7)
axes[1, 1].set_xlabel('特徴量インデックス')
axes[1, 1].set_ylabel('係数値')
axes[1, 1].set_title(f'推定係数の比較 (最初の{p_true}個が真に有効)')
axes[1, 1].legend()
axes[1, 1].axvline(p_true - 0.5, color='red', linestyle=':', alpha=0.7)

plt.suptitle('ベイズ推定（MAP） ≡ 正則化つき回帰\nRidge=ガウス事前分布 / LASSO=ラプラス事前分布',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('bridge_regularization.png', dpi=100, bbox_inches='tight')
plt.show()

n_zeros_lasso = np.sum(lasso_best.coef_ == 0)
print(f'\nLASSO がゼロにした係数: {n_zeros_lasso}/{p_total}  ← スパース推定')
print(f'（不要な特徴量 {p_total - p_true} 個のうち {n_zeros_lasso} 個が自動的に0に）')

---
## 4. 仮説検定 → モデル評価指標

### 4.1 対応関係

| 統計的仮説検定 | ML モデル評価 | 共通の本質 |
|--------------|--------------|----------|
| 帰無仮説 $H_0$ の棄却 | テスト精度 > ベースライン | 偶然を超えているか |
| 第一種の過誤（偽陽性率）$\alpha$ | 精度 (Precision), FPR | 誤りの許容水準 |
| 第二種の過誤（偽陰性率）$\beta$ | 再現率 (Recall), FNR | 見逃しの許容水準 |
| 検出力 $1-\beta$ | 再現率 (Recall) | 真の効果を検出する能力 |
| 効果量 (Effect size) | $R^2$, AUC, $F_1$ | 実際の強さ・有用性 |
| 交差検定（省略形） | 交差検証 (CV) | 汎化性能の評価 |

### 4.2 混同行列とエラー率

統計の「第一種・第二種の過誤」は分類の「FP・FN」と完全に対応します：

$$\text{Recall} = \frac{TP}{TP+FN} = 1 - \beta \quad (\text{検出力})$$

$$\text{FPR} = \frac{FP}{FP+TN} = \alpha \quad (\text{有意水準})$$

In [ ]:
from sklearn.metrics import classification_report, precision_recall_curve, roc_curve

# 分類モデルで第一種・第二種の過誤を可視化
y_prob = log_reg.predict_proba(X_te)[:, 1]
y_pred_cls = log_reg.predict(X_te)

cm = confusion_matrix(y_te, y_pred_cls)
TN, FP, FN, TP = cm.ravel()

print('=== 混同行列と統計の対応 ===')
print(f'\n             予測: 陰性  予測: 陽性')
print(f'実際: 陰性     {TN:4d}(TN)    {FP:4d}(FP)  ← 第一種の過誤 = FP/(FP+TN) = {FP/(FP+TN):.3f}')
print(f'実際: 陽性     {FN:4d}(FN)    {TP:4d}(TP)  ← 第二種の過誤 = FN/(FN+TP) = {FN/(FN+TP):.3f}')
print(f'\n検出力 (1-β) = Recall = {TP/(TP+FN):.3f}')
print(f'有意水準対応  (α)  = FPR   = {FP/(FP+TN):.3f}')
print()
print(classification_report(y_te, y_pred_cls, target_names=['クラス0', 'クラス1']))

In [ ]:
# 交差検証：統計の「繰り返し実験」に対応
cv_scores = cross_val_score(log_reg, X_cls, y_cls, cv=10, scoring='accuracy')

# t検定で「モデルがランダムより有意に良いか」を検定
baseline_acc = y_cls.mean()  # 多数決ベースライン
t_stat, p_value = stats.ttest_1samp(cv_scores, popmean=baseline_acc)

print('=== 交差検証 + 統計的有意性検定 ===')
print(f'10-fold CV 精度: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'ランダムベースライン: {baseline_acc:.4f}')
print(f'\n1標本t検定: t={t_stat:.3f}, p={p_value:.4f}')
if p_value < 0.05:
    print('→ p < 0.05: モデルはランダムより有意に優れている ✓')
else:
    print('→ p >= 0.05: 有意差なし')

# 95%信頼区間（統計）= CVの不確実性推定（ML）
ci = stats.t.interval(0.95, df=len(cv_scores)-1,
                       loc=cv_scores.mean(),
                       scale=stats.sem(cv_scores))
print(f'\n95%信頼区間: ({ci[0]:.4f}, {ci[1]:.4f})')

In [ ]:
# ROC 曲線 と 精度-再現率曲線
fpr_arr, tpr_arr, thresh_roc = roc_curve(y_te, y_prob)
prec_arr, rec_arr, thresh_pr = precision_recall_curve(y_te, y_prob)
auc_roc = roc_auc_score(y_te, y_prob)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ROC曲線
axes[0].plot(fpr_arr, tpr_arr, color='steelblue', linewidth=2, label=f'AUC={auc_roc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='ランダム (AUC=0.5)')
axes[0].fill_between(fpr_arr, tpr_arr, alpha=0.1, color='steelblue')
axes[0].set_xlabel('FPR = 偽陽性率 (= α)')
axes[0].set_ylabel('TPR = 再現率 (= 1-β)')
axes[0].set_title('ROC 曲線\n（α vs 1-β のトレードオフ）')
axes[0].legend()

# 精度-再現率曲線
axes[1].plot(rec_arr, prec_arr, color='darkorange', linewidth=2)
axes[1].set_xlabel('再現率 (Recall = 1-β)')
axes[1].set_ylabel('精度 (Precision)')
axes[1].set_title('精度-再現率曲線\n（クラス不均衡時に重要）')

# 混同行列の可視化
im = axes[2].imshow(cm, cmap='Blues')
axes[2].set_xticks([0, 1]); axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(['予測: 陰性', '予測: 陽性'])
axes[2].set_yticklabels(['実際: 陰性', '実際: 陽性'])
labels_cm = [['TN\n(正解)', 'FP\n(第一種の過誤)'],
              ['FN\n(第二種の過誤)', 'TP\n(正解)']]
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, f'{labels_cm[i][j]}\n{cm[i, j]}',
                     ha='center', va='center', fontsize=10,
                     color='white' if cm[i, j] > cm.max()/2 else 'black')
axes[2].set_title('混同行列\n（統計の過誤との対応）')
plt.colorbar(im, ax=axes[2])

plt.suptitle('仮説検定 ↔ ML評価指標の対応', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bridge_hypothesis_test.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 5. 総まとめ：統計学 ↔ 機械学習 対応表

In [ ]:
# 対応表を DataFrame で整理
bridge_table = pd.DataFrame({
    '統計学の概念': [
        '最小二乗法（OLS）',
        '最尤推定（MLE）',
        'MAP推定（ガウス事前分布）',
        'MAP推定（ラプラス事前分布）',
        '交差検定・ブートストラップ',
        '第一種の過誤（α）',
        '第二種の過誤（β）',
        '検出力（1-β）',
        '効果量・決定係数 R²',
        '多変量解析（主成分分析）',
        '判別分析（LDA）',
        '回帰診断・残差分析',
    ],
    '機械学習の概念': [
        '線形回帰（MSE最小化）',
        '損失関数最小化（全般）',
        'Ridge 回帰（L₂正則化）',
        'LASSO 回帰（L₁正則化）',
        '交差検証（CV）',
        '偽陽性率（FPR）',
        '偽陰性率（FNR）',
        '再現率（Recall）',
        'AUC / F₁スコア',
        '次元削減（PCA / オートエンコーダ）',
        '線形分類（ロジスティック回帰 / SVM）',
        '特徴量重要度・説明可能AI（SHAP）',
    ],
    '共通の本質': [
        '残差二乗和の最小化',
        'データの生成モデルを推定',
        '係数を0方向に縮小',
        'スパース推定（不要な係数を0に）',
        '汎化性能・信頼区間の推定',
        '陰性を陽性と誤分類する割合',
        '陽性を陰性と見逃す割合',
        '真の陽性を正しく検出する割合',
        '実用的な意義・有用性の指標',
        '分散を最大化する方向への変換',
        'クラス間の分離超平面を学習',
        '各変数が予測に与える影響の定量化',
    ]
})

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 20)
print('=== 統計学 ↔ 機械学習 概念対応表 ===')
print(bridge_table.to_string(index=False))

In [ ]:
# 概念マップの可視化
fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111)
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

# 左列（統計学）
stat_items = ['OLS\n最小二乗法', '最尤推定\n(MLE)', 'MAP推定\n(ガウス事前)',
              'MAP推定\n(ラプラス事前)', '第一種の過誤\nα', '検出力\n1-β', '交差検定']
# 右列（ML）
ml_items   = ['線形回帰\nMSE最小化', '損失関数\n最小化', 'Ridge\nL₂正則化',
              'LASSO\nL₁正則化', '偽陽性率\nFPR', '再現率\nRecall', '交差検証\nCV']
# 色
stat_color = '#3498db'
ml_color   = '#2ecc71'

y_positions = np.linspace(7, 0.5, len(stat_items))

for i, (s, m, y) in enumerate(zip(stat_items, ml_items, y_positions)):
    # 統計
    ax.text(1.5, y, s, ha='center', va='center', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=stat_color, alpha=0.3))
    # 矢印
    ax.annotate('', xy=(5.5, y), xytext=(3, y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    ax.text(4.25, y + 0.2, '≡', ha='center', va='center', fontsize=14, color='red')
    # ML
    ax.text(7.5, y, m, ha='center', va='center', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=ml_color, alpha=0.3))

# タイトル
ax.text(1.5, 7.7, '統計学', ha='center', fontsize=13, fontweight='bold', color=stat_color)
ax.text(7.5, 7.7, '機械学習 (ML)', ha='center', fontsize=13, fontweight='bold', color=ml_color)
ax.text(4.25, 7.7, '≡', ha='center', fontsize=16, color='red', fontweight='bold')

plt.title('統計学 ↔ 機械学習 概念マップ', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('bridge_concept_map.png', dpi=100, bbox_inches='tight')
plt.show()

---
## まとめ

### 核心メッセージ

> **統計学と機械学習は「同じ数学」を異なる目的で使っています。**

| 観点 | 統計学 | 機械学習 |
|------|--------|----------|
| **目的** | 母集団の理解・推測・仮説検定 | 未知データへの汎化・予測 |
| **重視** | 推定の解釈・係数の有意性 | テストセットの精度・実用性 |
| **前提** | 分布の仮定（正規性など） | 仮定を最小化、データで学ぶ |
| **評価** | p値・信頼区間・AIC/BIC | CV誤差・AUC・F₁スコア |

### 両者を理解することの強み

- **モデルの解釈**：係数の統計的有意性を確認しながら予測精度を高められる
- **正則化の根拠**：Ridge/LASSOが単なるトリックではなくベイズ的根拠があると理解できる
- **評価の深さ**：精度だけでなく第一種・第二種の過誤を意識した評価ができる
- **汎化性能**：交差検証を統計的検定と組み合わせてより信頼性の高い評価ができる

### 次のステップ
- **statspg 08**: 統計的推定の詳細（信頼区間・点推定）
- **mlpg**: scikit-learn を使った実践的なMLパイプライン
- **ベイズ深層学習**: ニューラルネットへのベイズ推定の応用